# Multiple Linear Regression (End-to-End)

This notebook trains and evaluates a Multiple Linear Regression (MLR) model on a synthetic car dataset.

## 1. Imports

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D)

pd.set_option("display.max_columns", 100)


## 2. Load Dataset

In [ ]:

# Adjust path if you moved things around
DATA_PATH = "data/cars_synthetic.csv"
df = pd.read_csv(DATA_PATH)
df.head()


## 3. Quick EDA

In [ ]:

display(df.info())
display(df.describe(include="all"))

# Pairplot on selected columns
sns.pairplot(df[["mpg","horsepower","weight","displacement","acceleration"]], diag_kind="hist")
plt.show()

# Correlation heatmap (numeric only)
plt.figure()
sns.heatmap(df[["mpg","horsepower","weight","displacement","acceleration","model_year"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


## 4. Features & Target

In [ ]:

target = "mpg"
cat_features = ["origin", "cylinders"]
num_features = ["horsepower", "weight", "displacement", "acceleration", "model_year"]

X = df[cat_features + num_features].copy()
y = df[target].copy()

display(X.head())
display(y.head())


## 5. Train / Test Split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape


## 6. Preprocessing with ColumnTransformer + Pipeline

In [ ]:

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_features),
        ("num", "passthrough", num_features),
    ],
    remainder="drop",
)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LinearRegression())
])

pipe


## 7. Train the Model

In [ ]:

pipe.fit(X_train, y_train)


## 8. Evaluation (R², RMSE, MAE)

In [ ]:

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

y_pred_train = pipe.predict(X_train)
y_pred_test  = pipe.predict(X_test)

print("=== PERFORMANCE ===")
print(f"Train R^2 : {r2_score(y_train, y_pred_train):.3f}")
print(f"Test  R^2 : {r2_score(y_test,  y_pred_test):.3f}")
print(f"Test  RMSE: {rmse(y_test, y_pred_test):.3f}")
print(f"Test  MAE : {mean_absolute_error(y_test, y_pred_test):.3f}")


## 9. Coefficients with Feature Names

In [ ]:

feature_names = pipe.named_steps["preprocess"].get_feature_names_out()
coefs = pipe.named_steps["model"].coef_
intercept = pipe.named_steps["model"].intercept_

coef_df = (pd.DataFrame({"feature": feature_names, "coef": coefs})
           .sort_values(by="coef", key=lambda s: s.abs(), ascending=False)
           .reset_index(drop=True))

print(f"Intercept: {intercept:.3f}")
display(coef_df.head(20))


## 10. Residual Diagnostics

In [ ]:

residuals = y_test - y_pred_test

# Residuals vs Predicted
plt.figure()
plt.scatter(y_pred_test, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted MPG")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted")
plt.show()

# Histogram of residuals
plt.figure()
plt.hist(residuals, bins=20)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Residuals Distribution")
plt.show()

# QQ plot for normality
sm.qqplot(residuals, line='45')
plt.title("QQ Plot of Residuals")
plt.show()


## 11. Multicollinearity (VIF) on Transformed Features

In [ ]:

# Get transformed train design matrix (without the target)
X_train_trans = pipe.named_steps["preprocess"].fit_transform(X_train)
X_train_trans = pd.DataFrame(X_train_trans, columns=pipe.named_steps["preprocess"].get_feature_names_out())

# Compute VIF (can be computationally heavy if many columns)
vif_df = pd.DataFrame({
    "feature": X_train_trans.columns,
    "VIF": [variance_inflation_factor(X_train_trans.values, i) for i in range(X_train_trans.shape[1])]
}).sort_values(by="VIF", ascending=False)

display(vif_df.head(20))


## 12. 3D Regression Plane (using two numeric features for visualization)

In [ ]:

# Train a simple model on two features (for visualization only)
vis_features = ["horsepower", "weight"]
X2 = df[vis_features].values
y2 = df["mpg"].values

lr2 = LinearRegression().fit(X2, y2)

x_s = np.linspace(X2[:,0].min(), X2[:,0].max(), 30)
y_s = np.linspace(X2[:,1].min(), X2[:,1].max(), 30)
xx, yy = np.meshgrid(x_s, y_s)
zz = lr2.intercept_ + lr2.coef_[0]*xx + lr2.coef_[1]*yy

fig = plt.figure(figsize=(9,6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X2[:,0], X2[:,1], y2, alpha=0.5)
ax.plot_surface(xx, yy, zz, alpha=0.4)
ax.set_xlabel("horsepower")
ax.set_ylabel("weight")
ax.set_zlabel("mpg")
ax.set_title("Regression Plane (2 features for viz)")
plt.show()


## 13. Statsmodels Summary (Optional)

In [ ]:

# Build a statsmodels OLS with encoded categoricals via patsy-style formula
df_copy = df.copy()
formula = "mpg ~ C(origin) + C(cylinders) + horsepower + weight + displacement + acceleration + model_year"
sm_model = sm.OLS.from_formula(formula, data=df_copy).fit()
print(sm_model.summary())
